# [v1] dokkaebi-ai 함수 단위 테스트 노트북

- 역할: **server 개발 전에 AI 함수/노드/그래프를 개별 테스트**
- 파이프라인 위치: AI 백엔드 검증용 (런타임 아님)
- 구현일: 2026-06-10 | 작성: kys (base-pipeline/kys/v1)

> 커널 = repo의 `.venv` 선택. 미설치면 아래 셀의 pip 주석 해제. 셀은 위→아래 순서로 실행.

In [1]:
# 0. 경로 설정 + (필요시) 설치
# !pip install -r ../requirements.txt
import os, sys
for cand in [os.getcwd(), os.path.join(os.getcwd(), ".."), os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(cand, "app")):
        root = os.path.abspath(cand)
        if root not in sys.path:
            sys.path.insert(0, root)
        print("repo root:", root)
        break
else:
    print("app/ 를 못 찾음 — repo 루트나 notebooks/ 에서 실행하세요")

repo root: /Users/gim-yeseul/Documents/2025_김예슬/2026-tourism-rpg/dokkaebi-ai


## 1. 설정(config) 확인 — 세마포어·재시도 수치

In [2]:
from app.config import get_settings
s = get_settings()
print("provider     :", s.llm_provider)
print("semaphore    :", s.llm_semaphore)
print("max_retries  :", s.llm_max_retries)
print("backoff base :", s.llm_backoff_base)
print("region cache :", s.region_cache_max)

provider     : mock
semaphore    : 8
max_retries  : 5
backoff base : 0.5
region cache : 8


## 2. 지역 인메모리 캐시 — 워밍 & 조회 (정찬희/이지선)

In [3]:
from app.region.memory_cache import get_region_cache
cache = get_region_cache()
cache.warm("jongno", {
    "jongno_unhyeongung": "운현궁은 흥선대원군이 살던 저택으로, 조선 말기 정치의 중심지였다.",
    "jongno_gwanghwamun": "광화문은 경복궁의 정문이며 현판으로 유명하다.",
})
print(cache.get_text("jongno_unhyeongung"))
print(cache.get_text("없는노드"))  # None (미스)

운현궁은 흥선대원군이 살던 저택으로, 조선 말기 정치의 중심지였다.
None


## 3. LLM 클라이언트 — mock 단일/병렬 호출 (정찬희)

In [4]:
from app.llm.client import LLMClient
llm = LLMClient()  # config의 provider=mock
print(await llm.generate("테스트 프롬프트"))
print(await llm.generate_many(["p1", "p2", "p3"]))  # 병렬(세마포어 공유)

[mock 도깨비] 허허, 아직 진짜 LLM이 붙지 않았느니라.
['[mock 도깨비] 허허, 아직 진짜 LLM이 붙지 않았느니라.', '[mock 도깨비] 허허, 아직 진짜 LLM이 붙지 않았느니라.', '[mock 도깨비] 허허, 아직 진짜 LLM이 붙지 않았느니라.']


## 4. LLM 429 백오프 재시도 검증
실패를 N번 던지는 가짜 provider로 지수 백오프가 실제로 도는지 확인.

In [5]:
import time
from app.llm.base import LLMProvider
from app.llm.client import LLMClient
from app.core.exceptions import LLMRateLimitError

class FlakyProvider(LLMProvider):
    # 처음 fail_times 번은 429, 그 다음 성공
    def __init__(self, fail_times=2):
        self.fail_times = fail_times
        self.calls = 0
    async def generate(self, prompt, **kw):
        self.calls += 1
        if self.calls <= self.fail_times:
            raise LLMRateLimitError("429 mock")
        return f"성공(시도 {self.calls}회)"

flaky = FlakyProvider(fail_times=2)
client = LLMClient(provider=flaky)
t0 = time.perf_counter()
out = await client.generate("429 테스트")
print(out, "| 총 호출", flaky.calls, "회 |", round(time.perf_counter() - t0, 2), "s (백오프 포함)")

2026-06-10 09:50:13,511 | WARNING | app.llm.client | LLM 429 → 0.53s 후 재시도 (1/5)


2026-06-10 09:50:14,040 | WARNING | app.llm.client | LLM 429 → 1.09s 후 재시도 (2/5)


성공(시도 3회) | 총 호출 3 회 | 1.63 s (백오프 포함)


## 5. 개별 노드 테스트 — 노드 함수 직접 호출
각 노드는 state(dict)를 받아 갱신할 일부를 반환. (2번 셀에서 지역 워밍 먼저)

In [6]:
from app.pipeline.nodes.persona_inject import persona_inject
from app.pipeline.nodes.context_load import context_load
from app.pipeline.nodes.prompt_assemble import prompt_assemble
from app.pipeline.nodes.generate import generate

state = {"node_id": "jongno_unhyeongung", "stage": "등장", "player_state": {}}
state.update(await persona_inject(state))
print("persona_inject  ->", state.get("cache_key"))
state.update(await context_load(state))
print("context_load    ->", state.get("context"))
state.update(await prompt_assemble(state))
print("prompt_assemble ->")
print(state.get("prompt"))
state.update(await generate(state))
print("generate        ->", state.get("response"))

persona_inject  -> npc:jongno_unhyeongung:등장
context_load    -> 운현궁은 흥선대원군이 살던 저택으로, 조선 말기 정치의 중심지였다.
prompt_assemble ->
[persona] {}
[stage] 등장
[장소 정보] 운현궁은 흥선대원군이 살던 저택으로, 조선 말기 정치의 중심지였다.
규칙: 위 장소 정보에 근거해서만, 도깨비 말투로 2~4문장.
generate        -> [mock 도깨비] 허허, 아직 진짜 LLM이 붙지 않았느니라.


## 6. 전체 그래프 E2E (services.run_dialogue)

In [7]:
from app.services.dialogue_service import run_dialogue
text, hit = await run_dialogue("jongno_unhyeongung", "등장", {})
print("cache_hit:", hit)
print("response :", text)

cache_hit: False
response : [mock 도깨비] 허허, 아직 진짜 LLM이 붙지 않았느니라.


## 7. (팀원용) 내 노드 채우고 바로 검증
- **이지선**: `persona_inject` persona 시드 로드 → 5번 셀로 확인
- **박준형**: `retrieve` / `prompt_assemble` / `generate` 품질 → 5·6번 셀
- **정찬희**: `cache` / `context_load` 배선 → 2·6번 셀

> 노드 파일 수정 후 **커널 재시작**(모듈 재import) → 위 셀 재실행.